### Trying out tsfresh

This notebook is just full of AI-slop

In [13]:
import numpy as np
import pandas as pd

from tsfresh import extract_features
from tsfresh.feature_extraction import EfficientFCParameters

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [14]:
# Combining the datasets together with a new column for run_id and road_id

dfA1 = pd.read_csv("../dataset/cleaned/Iisingintie0.csv")
dfA2 = pd.read_csv("../dataset/cleaned/Iisingintie1.csv")
dfA3 = pd.read_csv("../dataset/cleaned/Iisingintie2.csv")

dfB1 = pd.read_csv("../dataset/cleaned/Sinetänsalmentie0.csv")
dfB2 = pd.read_csv("../dataset/cleaned/Sinetänsalmentie1.csv")
dfB3 = pd.read_csv("../dataset/cleaned/Sinetänsalmentie2.csv")
dfB4 = pd.read_csv("../dataset/cleaned/Sinetänsalmentie3.csv")

dfC1 = pd.read_csv("../dataset/cleaned/Kaittiaisentie0.csv")


runs = {
    "A1": (dfA1, "A"), "A2": (dfA2, "A"), "A3": (dfA3, "A"),
    "B1": (dfB1, "B"), "B2": (dfB2, "B"), "B3": (dfB3, "B"), "B4": (dfB4, "B"),
    # "C1": (dfC1, "C"),
}

df = pd.concat(
    [d.assign(run_id=rid, road_id=road) for rid, (d, road) in runs.items()],
    ignore_index=True,
)

In [15]:
df["AccMagnitude"] = np.sqrt(
    df["AccX"]**2 + df["AccY"]**2 + df["AccZ"]**2
)

df["GyrMagnitude"] = np.sqrt(
    df["GyrX"]**2 + df["GyrY"]**2 + df["GyrZ"]**2
)

feature_cols = [
    "AccX", "AccY", "AccZ",
    "GyrX", "GyrY", "GyrZ",
    "Pitch(deg)", "Roll(deg)",
    "Speed_kmh", "RBCSV_80(cf)",
    "AccMagnitude", "GyrMagnitude"
]


## Tsfresh starts here

In [16]:
# Creating windows

WIN = 200
STRIDE = 150
MIN_PERCENTAGE = 0.1

long_data = []
window_labels = []
window_run_ids = []

window_id = 0

for run_name, run_df in df.groupby("run_id", sort=False):
    run_df = run_df.sort_values("Timer(ms)")

    X = run_df[feature_cols].to_numpy(dtype=np.float32)
    y = run_df["Frostbumps"].to_numpy()

    for start in range(0, len(X) - WIN + 1, STRIDE):
        end = start + WIN
        window = X[start:end]

        # One label for this whole window
        bump_percentage = y[start:end].mean()
        window_labels.append(int(bump_percentage >= MIN_PERCENTAGE))
        window_run_ids.append(run_name)

        # Convert this window to tsfresh's long format
        for i, col in enumerate(feature_cols):
            long_data.append(
                pd.DataFrame({
                    "id": window_id,
                    "time": np.arange(WIN),
                    "kind": col,
                    "value": window[:, i]
                })
            )

        window_id += 1

long_df = pd.concat(long_data, ignore_index=True)

print("Long dataframe shape:", long_df.shape)
print("Number of windows:", window_id)
print("Number of labels:", len(window_labels))
print("Number of run IDs:", len(window_run_ids))

Long dataframe shape: (4600800, 4)
Number of windows: 1917
Number of labels: 1917
Number of run IDs: 1917


Extract tsfresh features
Each window becomes one row. tsfresh calculates descriptive features for each sensor signal inside that window.

In [17]:
# Extracting tsfresh features

settings = EfficientFCParameters()

X_tsfresh = extract_features(
    long_df,
    column_id="id",
    column_sort="time",
    column_kind="kind",
    column_value="value",
    default_fc_parameters=settings,
    n_jobs=4,
    disable_progressbar=False
)

print("Extracted features:", X_tsfresh.shape)

Feature Extraction: 100%|██████████| 20/20 [04:51<00:00, 14.55s/it]


Extracted features: (1917, 9324)


Create labels and run IDs aligned to the feature rows
The id used by tsfresh is the window ID. We use it to line up the labels and run IDs with the extracted feature matrix.

In [18]:
window_metadata = pd.DataFrame({
    "Frostbumps": window_labels,
    "run_id": window_run_ids
})
window_metadata.index.name = "id"

# Align metadata to the exact row order returned by tsfresh
window_metadata = window_metadata.reindex(X_tsfresh.index)

assert window_metadata["Frostbumps"].notna().all()
assert window_metadata["run_id"].notna().all()

y_tsfresh = window_metadata["Frostbumps"].astype(int)
run_ids = window_metadata["run_id"]

print("Labels:", y_tsfresh.shape)
print("Run IDs:", run_ids.shape)
print("\nWindows per run:")
print(run_ids.value_counts().sort_index())
print("\nPositive label percentage:", y_tsfresh.mean())

Labels: (1917,)
Run IDs: (1917,)

Windows per run:
run_id
A1    378
A2    502
A3    764
B1     42
B2     50
B3     61
B4    120
Name: count, dtype: int64

Positive label percentage: 0.24830464267083985


## Split by run

In [ ]:
train_runs = ["A3", "B3", "B4", "B1"]
val_runs = ["A2", "B2"]
test_runs = ["A1"]

train_mask = run_ids.isin(train_runs)
val_mask = run_ids.isin(val_runs)
test_mask = run_ids.isin(test_runs)

Xtr = X_tsfresh.loc[train_mask].copy()
Xva = X_tsfresh.loc[val_mask].copy()
Xte = X_tsfresh.loc[test_mask].copy()

ytr = y_tsfresh.loc[train_mask].copy()
yva = y_tsfresh.loc[val_mask].copy()
yte = y_tsfresh.loc[test_mask].copy()

print("Train:", Xtr_raw.shape, "| positive:", f"{ytr.mean():.1%}")
print("Validation:", Xva_raw.shape, "| positive:", f"{yva.mean():.1%}")
print("Test:", Xte_raw.shape, "| positive:", f"{yte.mean():.1%}")

Train: (987, 9324) | positive: 25.4%
Validation: (552, 9324) | positive: 24.5%
Test: (378, 9324) | positive: 23.8%


Select features with RFE
RFE repeatedly trains an estimator and removes less important features. Here we keep 40 features.

This step uses training data and training labels only.

In [ ]:
rfe_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2,
    eval_metric="aucpr",
    random_state=69,
    n_jobs=-1
)

N_FEATURES_TO_KEEP = 40

rfe = RFE(
    estimator=rfe_model,
    n_features_to_select=N_FEATURES_TO_KEEP,
    step=0.1,
    verbose=1
)

rfe.fit(Xtr, ytr)

selected_features = Xtr.columns[rfe.support_]

print("Features before RFE:", Xtr.shape[1])
print("Features after RFE:", len(selected_features))
print("\nSelected features:")
print(selected_features.tolist())

Fitting estimator with 8774 features.
Fitting estimator with 7897 features.
Fitting estimator with 7020 features.
Fitting estimator with 6143 features.
Fitting estimator with 5266 features.
Fitting estimator with 4389 features.
Fitting estimator with 3512 features.
Fitting estimator with 2635 features.
Fitting estimator with 1758 features.
Fitting estimator with 881 features.
Features before RFE: 8774
Features after RFE: 40

Selected features:
['Speed_kmh__variation_coefficient', 'Speed_kmh__sum_of_reoccurring_values', 'AccMagnitude__autocorrelation__lag_8', 'AccX__fft_coefficient__attr_"abs"__coeff_0', 'AccX__number_crossing_m__m_0', 'AccY__fft_coefficient__attr_"angle"__coeff_87', 'AccZ__fft_coefficient__attr_"abs"__coeff_1', 'AccZ__agg_linear_trend__attr_"intercept"__chunk_len_5__f_agg_"max"', 'AccZ__agg_linear_trend__attr_"stderr"__chunk_len_10__f_agg_"var"', 'GyrMagnitude__c3__lag_3', 'GyrMagnitude__agg_autocorrelation__f_agg_"mean"__maxlag_40', 'GyrMagnitude__change_quantiles__f_

INspecting RFE rankings

In [ ]:
rfe_results = pd.DataFrame({
    "Feature": Xtr.columns,
    "Selected": rfe.support_,
    "Ranking": rfe.ranking_
}).sort_values(["Selected", "Ranking"], ascending=[False, True])

display(rfe_results)

,Feature,Selected,Ranking
12,Speed_kmh__variation_coefficient,True,1
28,Speed_kmh__sum_of_reoccurring_values,True,1
790,AccMagnitude__autocorrelation__lag_8,True,1
1874,"AccX__fft_coefficient__attr_""abs""__coeff_0",True,1
2144,AccX__number_crossing_m__m_0,True,1
...,...,...,...
1020,"AccMagnitude__fft_coefficient__attr_""real""__co...",False,11
1021,"AccMagnitude__fft_coefficient__attr_""real""__co...",False,11
1022,"AccMagnitude__fft_coefficient__attr_""real""__co...",False,11
1024,"AccMagnitude__fft_coefficient__attr_""real""__co...",False,11


10. Build selected-feature datasets
Apply the same selected feature names to train, validation and test.

In [ ]:
Xtr_selected = Xtr[selected_features]
Xva_selected = Xva[selected_features]
Xte_selected = Xte[selected_features]

print(Xtr_selected.shape, Xva_selected.shape, Xte_selected.shape)

(987, 40) (552, 40) (378, 40)


Training XGBoost

In [24]:
xgb_tsfresh = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2,
    eval_metric="aucpr",
    random_state=69,
    n_jobs=-1
)

xgb_tsfresh.fit(Xtr_selected, ytr)

val_prob = xgb_tsfresh.predict_proba(Xva_selected)[:, 1]

print("Validation PR-AUC:", average_precision_score(yva, val_prob))
print("Validation ROC-AUC:", roc_auc_score(yva, val_prob))

Validation PR-AUC: 0.7107160389198619
Validation ROC-AUC: 0.8550492938982148


In [25]:
test_prob = xgb_tsfresh.predict_proba(Xte_selected)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print("Test PR-AUC:", average_precision_score(yte, test_prob))
print("Test ROC-AUC:", roc_auc_score(yte, test_prob))

print("\nConfusion matrix:")
print(confusion_matrix(yte, test_pred))

print("\nClassification report:")
print(classification_report(yte, test_pred, zero_division=0))

Test PR-AUC: 0.6348681224232242
Test ROC-AUC: 0.8321759259259259

Confusion matrix:
[[246  42]
 [ 45  45]]

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.85      0.85       288
           1       0.52      0.50      0.51        90

    accuracy                           0.77       378
   macro avg       0.68      0.68      0.68       378
weighted avg       0.77      0.77      0.77       378

